In [7]:
# ----------------------------
# Clean + aggregate Generic Keywords (GKs) by Basic Type
# Input : New_BT_GKs_Raw.csv (2 columns: "Basic Type", "Generic keywords")
# Output: New_BT_GKs_Cleaned_Unique.csv (2 columns: "Basic Type", "GKs")
#         where GKs is a Python-list-like string using single quotes:
#         ['kw1', 'kw2', ...]
# ----------------------------

import os
import re
import pandas as pd
from collections import OrderedDict

# ---------- 1) Locate input CSV (works in fresh notebooks) ----------
candidate_paths = [
    os.path.abspath("../Data/New_BT_GKs_Raw.csv"),        # relative to project root
    os.path.expanduser("~/Data/New_BT_GKs_Raw.csv"),   # user said home directory
    "/mnt/data/Data/New_BT_GKs_Raw.csv",               # common sandbox path
]

input_path = None
for p in candidate_paths:
    if os.path.exists(p):
        input_path = p
        break

if input_path is None:
    raise FileNotFoundError(
        "Couldn't find New_BT_GKs_Raw.csv. Tried:\n" + "\n".join(candidate_paths)
    )

print("✅ Reading:", input_path)

# ---------- 2) Read CSV robustly ----------
# (handles extra columns; we only keep the first two if needed)
df = pd.read_csv(input_path)

# Normalize column names (strip spaces)
df.columns = [c.strip() for c in df.columns]

# Try to find the required columns even if capitalization differs
def find_col(df_cols, target):
    target_norm = re.sub(r"\s+", " ", target.strip().lower())
    for c in df_cols:
        c_norm = re.sub(r"\s+", " ", c.strip().lower())
        if c_norm == target_norm:
            return c
    return None

bt_col = find_col(df.columns, "Basic Type")
gk_col = find_col(df.columns, "Generic keywords")

# If not found, fallback to first two columns
if bt_col is None or gk_col is None:
    if len(df.columns) < 2:
        raise ValueError("CSV must have at least 2 columns for Basic Type and Generic keywords.")
    bt_col, gk_col = df.columns[0], df.columns[1]
    print(f"⚠️ Column names not matched. Falling back to first two columns: {bt_col!r}, {gk_col!r}")
else:
    print(f"✅ Using columns: {bt_col!r}, {gk_col!r}")

df = df[[bt_col, gk_col]].copy()
df.rename(columns={bt_col: "Basic Type", gk_col: "Generic keywords"}, inplace=True)

# ---------- 3) Cleaning helpers ----------
def normalize_basic_type(x: str) -> str:
    if pd.isna(x):
        return ""
    # collapse whitespace + strip
    s = re.sub(r"\s+", " ", str(x)).strip()
    return s

def split_keywords(cell) -> list[str]:
    """
    Split a cell like:
      "Ceregrow, Baby Care, Baby Cereal, Nestle Ceregrow"
    into a list of cleaned keyword strings.
    """
    if pd.isna(cell):
        return []
    s = str(cell)

    # common cleanup: normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    # split on commas (simple + matches your sample)
    parts = [p.strip() for p in s.split(",")]

    # remove empties + normalize internal whitespace
    cleaned = []
    for p in parts:
        p2 = re.sub(r"\s+", " ", p).strip()
        if p2 != "":
            cleaned.append(p2)
    return cleaned

def ordered_unique_case_insensitive(items: list[str]) -> list[str]:
    """
    De-dup keywords case-insensitively while preserving first-seen order.
    Keeps the original casing of the first occurrence.
    """
    seen = set()
    out = []
    for it in items:
        key = it.casefold()
        if key not in seen:
            seen.add(key)
            out.append(it)
    return out

def to_single_quote_list_literal(items: list[str]) -> str:
    """
    Convert list -> string in format:
      ['a', 'b', 'c']
    Escapes single quotes inside items.
    """
    escaped = [i.replace("\\", "\\\\").replace("'", "\\'") for i in items]
    return "[" + ", ".join([f"'{i}'" for i in escaped]) + "]"

# ---------- 4) Clean + aggregate ----------
df["Basic Type"] = df["Basic Type"].map(normalize_basic_type)
df["Generic keywords"] = df["Generic keywords"].astype("string")

# Drop blank Basic Types
df = df[df["Basic Type"].ne("")].copy()

# Build aggregation with stable ordering:
# - preserve Basic Type order as first appearance in the file
# - preserve GK order as first appearance across that Basic Type
basic_type_order = df["Basic Type"].drop_duplicates().tolist()

gks_by_bt: dict[str, list[str]] = OrderedDict((bt, []) for bt in basic_type_order)

for bt, cell in zip(df["Basic Type"], df["Generic keywords"]):
    gks_by_bt[bt].extend(split_keywords(cell))

# De-dup per basic type
rows = []
for bt, gks in gks_by_bt.items():
    uniq_gks = ordered_unique_case_insensitive(gks)
    rows.append(
        {
            "Basic Type": bt,
            "GKs": to_single_quote_list_literal(uniq_gks),
            "_gk_count": len(uniq_gks),  # helper column for inspection (will drop before saving)
        }
    )

out_df = pd.DataFrame(rows)

# Optional: sort by Basic Type (comment out if you want original order)
# out_df = out_df.sort_values("Basic Type", kind="stable").reset_index(drop=True)

# ---------- 5) Save output ----------
input_dir = os.path.dirname(input_path) or "."
output_path = os.path.join(input_dir, "New_BT_GKs_Cleaned_Unique.csv")

out_df_to_save = out_df.drop(columns=["_gk_count"])
out_df_to_save.to_csv(output_path, index=False, encoding="utf-8")

print("✅ Saved:", output_path)

# ---------- 6) Quick sanity checks / preview ----------
print("\nPreview (first 20 rows):")
display(out_df_to_save.head(20))

print("\nTop 10 Basic Types by GK count:")
display(out_df.sort_values("_gk_count", ascending=False).head(10)[["Basic Type", "_gk_count"]])

✅ Reading: c:\Code\AI Tagging\Data\New_BT_GKs_Raw.csv
✅ Using columns: 'Basic Type', 'Generic keywords'
✅ Saved: c:\Code\AI Tagging\Data\New_BT_GKs_Cleaned_Unique.csv

Preview (first 20 rows):


,Basic Type,GKs
0,Baby Cereal,"['Ceregrow', 'Baby Care', 'Baby Cereal', 'Nest..."
1,Baby Milk Powder,"['Baby Care', 'Baby Milk Powder', 'Infant Baby..."
2,Biscuits,"['Biscuit', 'Groceries', 'Baby Rusks', 'Orange..."
3,Baby Detergent Liquid,"['Baby Care', 'Baby Laundry', 'Farlin baby det..."
4,Baby Body Wash,"['Baby Care', 'Baby Body Wash', 'Kates Choice ..."
5,Baby Bottle Brush,"['Baby Care', 'Baby Bottle Brush', 'Farlin Bot..."
6,Baby Cloth Pegs,"['Baby Care', 'Cloth Peg', 'Baby Cloth Pegs', ..."
7,Baby Cologne,"['Cologne', 'Baby Care', 'Baby Cologne', 'Flor..."
8,Baby Cotton Buds,"['Baby Care', 'Baby Cotton Buds', 'Kids Joy Co..."
9,Baby Cream,"['Baby Care', 'Baby Cream', 'Kates Choice Kids..."



Top 10 Basic Types by GK count:


,Basic Type,_gk_count
2,Biscuits,278
81,Chocolate,174
638,Sausage,106
305,Ice Cream Tub,106
685,Soya Meat,98
332,Face Wash,90
467,Pasta,83
72,Fruit Drink,83
523,Instant Noodles,81
80,Wafers,80


In [9]:
import os
import pandas as pd
import numpy as np

# =========================
# CONFIG (edit these)
# =========================
N_SAMPLE = 20                 # total number of SKU rows you want in the test sample
RANDOM_SEED = 42               # reproducible randomness
MIN_PER_CATEGORY = 1           # ensure at least this many rows per category (if possible)
MAX_PER_CATEGORY = None        # optionally cap rows per category (e.g., 200). Keep None to disable.
CATEGORY_COL = "Categories"    # column name in Catalog_SKUs.csv
NAME_COL = "Name"              # column name in Catalog_SKUs.csv

# Input path: 'Testing Data/Catalog_SKUs.csv'
candidate_paths = [
    os.path.join("Testing Data", "Catalog_SKUs.csv"),
    os.path.expanduser("~/Testing Data/Catalog_SKUs.csv"),
    os.path.abspath(os.path.join("Testing Data", "Catalog_SKUs.csv")),
    "/mnt/data/Testing Data/Catalog_SKUs.csv",
    "/mnt/data/Testing_Data/Catalog_SKUs.csv",
]

input_path = next((p for p in candidate_paths if os.path.exists(p)), None)
if input_path is None:
    raise FileNotFoundError(
        "Couldn't find Catalog_SKUs.csv. Tried:\n" + "\n".join(candidate_paths)
    )

print("✅ Reading:", input_path)

# =========================
# LOAD + CLEAN
# =========================
df = pd.read_csv(input_path)
df.columns = [c.strip() for c in df.columns]

if CATEGORY_COL not in df.columns or NAME_COL not in df.columns:
    raise ValueError(
        f"Expected columns '{NAME_COL}' and '{CATEGORY_COL}'. "
        f"Found: {list(df.columns)}"
    )

# Drop empty category/name rows
df = df.copy()
df[CATEGORY_COL] = df[CATEGORY_COL].astype("string").fillna("").str.strip()
df[NAME_COL] = df[NAME_COL].astype("string").fillna("").str.strip()
df = df[(df[CATEGORY_COL] != "") & (df[NAME_COL] != "")].reset_index(drop=True)

# If Categories contains multiple values separated by comma/pipe/etc.,
# normalize by taking the FIRST category as the sampling stratum.
# (Adjust split regex if your Categories column uses a different delimiter.)
df["_category_stratum"] = df[CATEGORY_COL].str.split(r"\s*[,|/]\s*", regex=True).str[0].str.strip()
df["_category_stratum"] = df["_category_stratum"].replace("", np.nan)
df = df.dropna(subset=["_category_stratum"]).reset_index(drop=True)

total_available = len(df)
if N_SAMPLE > total_available:
    print(f"⚠️ Requested N_SAMPLE={N_SAMPLE} but only {total_available} rows available. Using all rows.")
    N_SAMPLE = total_available

# =========================
# STRATIFIED SAMPLING LOGIC
# - Base: proportional to category sizes
# - Enforce MIN_PER_CATEGORY where possible
# - Optionally enforce MAX_PER_CATEGORY
# - Fill remainder by weighted sampling from remaining rows
# =========================
rng = np.random.default_rng(RANDOM_SEED)

cat_counts = df["_category_stratum"].value_counts()
cats = cat_counts.index.tolist()
k = len(cats)

# If N_SAMPLE is too small to satisfy MIN_PER_CATEGORY across all categories,
# reduce the effective min to fit.
effective_min = MIN_PER_CATEGORY
if k * effective_min > N_SAMPLE:
    effective_min = N_SAMPLE // k
    print(f"⚠️ N_SAMPLE too small for MIN_PER_CATEGORY={MIN_PER_CATEGORY} across {k} categories.")
    print(f"   Using effective MIN_PER_CATEGORY={effective_min}.")

# Initial allocation: proportional
proportions = cat_counts / cat_counts.sum()
alloc = (proportions * N_SAMPLE).round().astype(int)

# Enforce min where feasible
if effective_min > 0:
    alloc = alloc.clip(lower=effective_min)

# Enforce max if provided
if MAX_PER_CATEGORY is not None:
    alloc = alloc.clip(upper=int(MAX_PER_CATEGORY))

# Cap each category allocation to available rows in that category
alloc = pd.Series(
    {c: min(int(alloc.get(c, 0)), int(cat_counts[c])) for c in cats}
)

# Adjust total to exactly N_SAMPLE
current_total = int(alloc.sum())

def take_from_categories(alloc_series, needed):
    """Reduce allocation (if needed < 0) from categories with slack > 0."""
    if needed >= 0:
        return alloc_series

    needed = -needed
    # Reduce from categories that are above effective_min (or above 0 if effective_min is 0)
    floor = effective_min if effective_min > 0 else 0
    reducible = alloc_series - floor
    reducible = reducible[reducible > 0].sort_values(ascending=False)

    for c in reducible.index:
        if needed == 0:
            break
        dec = min(int(reducible[c]), needed)
        alloc_series[c] -= dec
        needed -= dec

    return alloc_series

def add_to_categories(alloc_series, needed):
    """Increase allocation up to availability (and MAX_PER_CATEGORY if set)."""
    if needed <= 0:
        return alloc_series

    # Remaining capacity per category
    cap = pd.Series({c: int(cat_counts[c]) for c in cats}) - alloc_series
    if MAX_PER_CATEGORY is not None:
        cap = np.minimum(cap, int(MAX_PER_CATEGORY) - alloc_series)
    cap = cap[cap > 0]

    # Distribute additions proportional to remaining capacity
    if cap.empty:
        return alloc_series

    cap_props = cap / cap.sum()
    add = (cap_props * needed).round().astype(int)

    # Ensure we don't add more than capacity
    for c in add.index:
        add[c] = min(int(add[c]), int(cap[c]))

    # Fix rounding mismatch by greedy filling
    added = int(add.sum())
    remainder = needed - added

    alloc_series.loc[add.index] += add

    if remainder > 0:
        cap2 = pd.Series({c: int(cat_counts[c]) for c in cats}) - alloc_series
        if MAX_PER_CATEGORY is not None:
            cap2 = np.minimum(cap2, int(MAX_PER_CATEGORY) - alloc_series)
        cap2 = cap2[cap2 > 0].sort_values(ascending=False)

        for c in cap2.index:
            if remainder == 0:
                break
            alloc_series[c] += 1
            remainder -= 1

    return alloc_series

# First pass: fix over/under allocation to match N_SAMPLE
alloc = take_from_categories(alloc.copy(), current_total - N_SAMPLE)
alloc = add_to_categories(alloc.copy(), N_SAMPLE - int(alloc.sum()))
final_total = int(alloc.sum())

# If still not exact (rare corner cases), fallback to simple global sample
if final_total != N_SAMPLE:
    print(f"⚠️ Allocation could not reach exact N_SAMPLE. Falling back to simple random sample of {N_SAMPLE}.")
    sampled_df = df.sample(n=N_SAMPLE, random_state=RANDOM_SEED).drop(columns=["_category_stratum"])
else:
    # Sample per category
    parts = []
    for c, n in alloc.items():
        if n <= 0:
            continue
        sub = df[df["_category_stratum"] == c]
        parts.append(sub.sample(n=n, random_state=int(rng.integers(0, 2**31 - 1))))
    sampled_df = pd.concat(parts, ignore_index=True)

    # Shuffle final sample
    sampled_df = sampled_df.sample(frac=1, random_state=RANDOM_SEED).reset_index(drop=True)
    sampled_df = sampled_df.drop(columns=["_category_stratum"])

# =========================
# SAVE OUTPUT FILES
# =========================
out_dir = os.path.dirname(input_path) or "."
sample_csv_path = os.path.join(out_dir, f"Catalog_SKUs_TestSample_{N_SAMPLE}.csv")
names_csv_path = os.path.join(out_dir, f"Catalog_SKUs_TestSample_{N_SAMPLE}_NamesOnly.csv")

sampled_df.to_csv(sample_csv_path, index=False, encoding="utf-8")

# Names-only file (one column: Name)
sampled_df[[NAME_COL]].to_csv(names_csv_path, index=False, encoding="utf-8")

print("✅ Saved sample file:", sample_csv_path)
print("✅ Saved names-only file:", names_csv_path)

# =========================
# QUICK DISTRIBUTION CHECK
# =========================
dist = sampled_df[CATEGORY_COL].astype("string").fillna("").str.split(r"\s*[,|/]\s*", regex=True).str[0].str.strip()
dist = dist.value_counts().rename_axis("Category").reset_index(name="count")

print("\nSample distribution (by first category token):")
display(dist.head(30))
print(f"\nTotal sampled rows: {len(sampled_df)}")

✅ Reading: Testing Data\Catalog_SKUs.csv
⚠️ Allocation could not reach exact N_SAMPLE. Falling back to simple random sample of 20.
✅ Saved sample file: Testing Data\Catalog_SKUs_TestSample_20.csv
✅ Saved names-only file: Testing Data\Catalog_SKUs_TestSample_20_NamesOnly.csv

Sample distribution (by first category token):


,Category,count
0,Groceries,10
1,Personal Care,3
2,Dairy,3
3,Baby Care,2
4,Stationery,1
5,Frozen Foods,1



Total sampled rows: 20


In [12]:
# =========================
# CHECK TAGGING ACCURACY
# =========================
import pandas as pd

# CONFIG: File names (edit these as needed)
RESULT_FILE = "tagging_results.csv"  # File with tagging results
SAMPLE_FILE = "Catalog_SKUs_TestSample_10.csv"  # File with catalog sample

# Load the result and sample files
result_path = os.path.join(os.path.dirname(input_path), RESULT_FILE)
sample_path = os.path.join(os.path.dirname(input_path), SAMPLE_FILE)

result_df = pd.read_csv(result_path)
sample_df = pd.read_csv(sample_path)

# Normalize column names for consistency
result_df.columns = [c.strip().lower() for c in result_df.columns]
sample_df.columns = [c.strip().lower() for c in sample_df.columns]

# Ensure required columns exist
required_sample_columns = {"name", "categories", "basic type", "generic keywords"}
required_result_columns = {"sku name", "category", "basic type", "generic keywords"}

if not required_sample_columns.issubset(sample_df.columns):
    raise ValueError(f"Sample file must contain columns: {required_sample_columns}")

if not required_result_columns.issubset(result_df.columns):
    raise ValueError(f"Result file must contain columns: {required_result_columns}")

# Check tagging accuracy
accuracy_report = []
category_correct_count = 0
basic_type_correct_count = 0

for _, row in sample_df.iterrows():
    sku_name = row["name"]
    expected_category = row["categories"]
    expected_bt = row["basic type"]
    expected_gks = set(row["generic keywords"].strip("[]").replace("'", "").split(", "))

    # Find the corresponding result row
    result_row = result_df[result_df["sku name"] == sku_name]

    if result_row.empty:
        accuracy_report.append({
            "SKU": sku_name,
            "Category Correct": False,
            "Basic Type Correct": False,
            "Expected Category": expected_category,
            "Result Category": None,
            "Expected Basic Type": expected_bt,
            "Result Basic Type": None,
            "Missing GKs": list(expected_gks),
            "Extra GKs": [],
            "GK Accuracy (%)": 0.0
        })
        continue

    result_category = result_row.iloc[0]["category"]
    result_bt = result_row.iloc[0]["basic type"]
    result_gks = set(result_row.iloc[0]["generic keywords"].strip("[]").replace("'", "").split(", "))

    missing_gks = expected_gks - result_gks
    extra_gks = result_gks - expected_gks
    gk_accuracy = (len(expected_gks & result_gks) / len(expected_gks)) * 100 if expected_gks else 100.0

    category_correct = result_category == expected_category
    basic_type_correct = result_bt == expected_bt

    if category_correct:
        category_correct_count += 1
    if basic_type_correct:
        basic_type_correct_count += 1

    accuracy_report.append({
        "SKU": sku_name,
        "Category Correct": category_correct,
        "Basic Type Correct": basic_type_correct,
        "Expected Category": expected_category,
        "Result Category": result_category,
        "Expected Basic Type": expected_bt,
        "Result Basic Type": result_bt,
        "Missing GKs": list(missing_gks),
        "Extra GKs": list(extra_gks),
        "GK Accuracy (%)": gk_accuracy
    })

# Convert accuracy report to DataFrame
accuracy_df = pd.DataFrame(accuracy_report)

# Calculate overall accuracy
total_records = len(sample_df)
category_accuracy = (category_correct_count / total_records) * 100
basic_type_accuracy = (basic_type_correct_count / total_records) * 100

# Print summary results
print("\n=========================")
print("TAGGING ACCURACY SUMMARY")
print("=========================")
print(f"Total Records: {total_records}")
print(f"Category Accuracy: {category_accuracy:.2f}%")
print(f"Basic Type Accuracy: {basic_type_accuracy:.2f}%")
print("\nDetailed Results:")
print(accuracy_df)

# Save the accuracy report
accuracy_report_path = os.path.join(os.path.dirname(input_path), "Tagging_Accuracy_Report.csv")
accuracy_df.to_csv(accuracy_report_path, index=False, encoding="utf-8")

print("\n✅ Tagging accuracy report saved to:", accuracy_report_path)


TAGGING ACCURACY SUMMARY
Total Records: 10
Category Accuracy: 90.00%
Basic Type Accuracy: 40.00%

Detailed Results:
                                               SKU  Category Correct  \
0         Mortein Flying Insect Killer Lemon 250ml              True   
1         Ritzbury Revello Chocolate Hazelnut 100g              True   
2                            Gillette Fusion Razor              True   
3    Prima Noodles Stella Vegetable No Add Msg 74g              True   
4                    Evon Deo Roll On Natural 30ml              True   
5                      Nature Turmeric Powder 100g              True   
6                           Alerics Malt Crunch 1l             False   
7                          Haus Boom Grapple 325Ml              True   
8  Tong Garden Coated Peanuts Barbecue Flavour 50g              True   
9              Kumarika Shampoo Long & Black 180ml              True   

   Basic Type Correct Expected Category Result Category  \
0               False         H